In [ ]:
# -*- coding: utf-8 -*-
"""
Collecte des donnees meteo horaires (Open-Meteo Archive / ERA5)
et insertion dans SQL Server -> table staging_meteo.

Usage :
    python collecte_meteo.py                          # utilise les dates par defaut
    python collecte_meteo.py 2026-07-01 2026-08-31    # dates explicites

Le script est idempotent : il supprime la plage de dates concernee
avant d'inserer, donc il peut etre rejoue sans creer de doublons.
"""

import sys
import urllib.request
import urllib.parse
import json
import pyodbc

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------

SERVEUR = "ICT-202-11"
BASE = "TFE_STIB"

# Point unique, centre de la Region de Bruxelles-Capitale
LATITUDE = 50.8503
LONGITUDE = 4.3517
FUSEAU = "Europe/Brussels"   # CRITIQUE : sinon les donnees arrivent en UTC

DATE_DEBUT_DEFAUT = "2026-07-01"
DATE_FIN_DEFAUT = "2026-07-31"

URL_API = "https://archive-api.open-meteo.com/v1/archive"

# Variables horaires demandees a l'API.
# L'ordre de cette liste determine l'ordre des colonnes inserees.
VARIABLES = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "wind_speed_10m",
    "wind_gusts_10m",
    "cloud_cover",
]


# ----------------------------------------------------------------------
# 1. RECUPERATION DES DONNEES
# ----------------------------------------------------------------------

def recuperer_meteo(date_debut, date_fin):
    """Interroge l'API Open-Meteo et retourne le dictionnaire 'hourly'."""
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": date_debut,
        "end_date": date_fin,
        "hourly": ",".join(VARIABLES),
        "timezone": FUSEAU,
    }
    url = URL_API + "?" + urllib.parse.urlencode(params)
    print("Appel API :", url)

    with urllib.request.urlopen(url, timeout=60) as reponse:
        donnees = json.loads(reponse.read().decode("utf-8"))

    if "hourly" not in donnees:
        raise RuntimeError("Reponse inattendue de l'API : " + json.dumps(donnees)[:500])

    print("Fuseau retourne par l'API :", donnees.get("timezone"),
          "| decalage UTC :", donnees.get("utc_offset_seconds"), "s")

    return donnees["hourly"]


# ----------------------------------------------------------------------
# 2. TRANSFORMATION EN LIGNES
# ----------------------------------------------------------------------

def construire_lignes(hourly):
    """
    L'API renvoie des listes paralleles :
        {"time": [...], "temperature_2m": [...], "precipitation": [...], ...}
    On les transpose en lignes (une ligne = une heure).
    """
    horodatages = hourly["time"]
    nb = len(horodatages)
    lignes = []

    for i in range(nb):
        horodatage = horodatages[i]          # format "2026-07-01T14:00"
        date_partie, heure_partie = horodatage.split("T")

        ligne = [
            horodatage,
            date_partie,
            heure_partie,
        ]
        for variable in VARIABLES:
            valeur = hourly[variable][i]
            ligne.append(None if valeur is None else str(valeur))

        lignes.append(tuple(ligne))

    return lignes


# ----------------------------------------------------------------------
# 3. INSERTION EN BASE
# ----------------------------------------------------------------------

def inserer(lignes, date_debut, date_fin):
    chaine = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={SERVEUR};DATABASE={BASE};Trusted_Connection=yes;"
    )
    conn = pyodbc.connect(chaine)
    cur = conn.cursor()
    cur.fast_executemany = True

    # Idempotence : on purge la plage avant de reinserer
    cur.execute(
        "DELETE FROM staging_meteo WHERE date_locale >= ? AND date_locale <= ?",
        date_debut, date_fin,
    )
    print("Lignes supprimees sur la plage :", cur.rowcount)

    sql = """
        INSERT INTO staging_meteo (
            date_heure_local, date_locale, heure_locale,
            temperature_2m, apparent_temp, humidite_rel,
            precipitation_mm, pluie_mm, neige_cm,
            code_meteo, vent_kmh, rafales_kmh, couverture_nuage
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
    """
    cur.executemany(sql, lignes)
    conn.commit()

    cur.execute("SELECT COUNT(*) FROM staging_meteo")
    total = cur.fetchone()[0]

    cur.close()
    conn.close()
    return total


# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------

if __name__ == "__main__":
    if len(sys.argv) == 3:
        debut, fin = sys.argv[1], sys.argv[2]
    else:
        debut, fin = DATE_DEBUT_DEFAUT, DATE_FIN_DEFAUT

    print(f"Periode demandee : {debut} -> {fin}")

    hourly = recuperer_meteo(debut, fin)
    lignes = construire_lignes(hourly)
    print("Lignes construites :", len(lignes))
    print("Premiere ligne :", lignes[0])
    print("Derniere ligne :", lignes[-1])

    total = inserer(lignes, debut, fin)
    print("OK. Total de lignes dans staging_meteo :", total)